# Knowledge Retrieval RAG

Ground an agent's answer in retrieved documents or records.


## Core ideas

- Retrieval before generation: ground answers in documents, records, or search results.
- Chunking and indexing: retrieval quality depends on document segmentation and metadata.
- Ranking: lexical, vector, hybrid, and reranking approaches trade speed and relevance.
- Grounded synthesis: answers should cite retrieved source ids and avoid unsupported claims.
- Failure modes: stale docs, missing docs, bad chunks, prompt injection in documents, and over-reliance on weak evidence.


## Common failure modes

- Retrieving weak context and still answering confidently.
- No citations or source ids.
- Ignoring prompt injection inside documents.


## Example



## Alternative framework implementation

LlamaIndex is included because ingestion, indexing, retrieval, and query engines are its primary abstraction. This example uses a tiny in-memory corpus so each stage stays visible.


In [ ]:
# ALTERNATIVE FRAMEWORK EXAMPLE: LlamaIndex RAG
# Import the dependencies used by this example.
import os

if not os.getenv("OPENAI_API_KEY"):
    print("Skipped: set OPENAI_API_KEY to build embeddings and answer the query.")
else:
    from llama_index.core import Document, Settings, VectorStoreIndex
    from llama_index.embeddings.openai import OpenAIEmbedding
    from llama_index.llms.openai import OpenAI

    # Documents are the source records. Metadata travels with retrieved chunks.
    documents = [
        Document(text="Refunds are available for 30 days after purchase.", metadata={"source": "refund-policy"}),
        Document(text="Enterprise plans include SSO and audit logs.", metadata={"source": "enterprise-plan"}),
    ]

    Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
    Settings.llm = OpenAI(model="gpt-5.6-sol")

    # The index embeds documents; the query engine retrieves relevant chunks and
    # asks the model to answer from them.
    index = VectorStoreIndex.from_documents(documents)
    query_engine = index.as_query_engine(similarity_top_k=2)
    response = query_engine.query("How long do customers have to request a refund?")

    print(response)
    print([node.metadata["source"] for node in response.source_nodes])


In [ ]:
docs = {
    "doc-1": "Prompt chaining decomposes complex tasks into ordered steps.",
    "doc-2": "Routing sends a task to a specialized handler.",
    "doc-3": "Memory stores useful context across interactions.",
}

# Define the data shape and small operations before running them.
def retrieve(query, k=2):
    q = set(query.lower().split())
    scored = []
    for doc_id, text in docs.items():
        score = len(q & set(text.lower().split()))
        scored.append((score, doc_id, text))
    return [(doc_id, text) for score, doc_id, text in sorted(scored, reverse=True)[:k] if score]

retrieve("How do agents keep context in memory?")
